In [ ]:
!pip install -q langgraph langchain_groq opencv-python-headless pillow pyngrok sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00


In [5]:
!pip install -q -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.3 MB/s eta 0:00:00


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.2 MB/s eta 0:00:00


In [13]:
%%writefile app.py
import streamlit as st
import base64
import base64
import cv2
import numpy as np
import io
from typing import TypedDict, List, Optional
from PIL import Image
#from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
import os
from google.colab import userdata



# 1. State Definition
class CoffeeReaderState(TypedDict):
    raw_images: List[str]
    clean_images: List[str]
    reading: str
    critique_notes: Optional[str]
    iteration_count: int


# 2. Nodes
def cleaner_node(state: CoffeeReaderState):
    processed = []
    for b64 in state["raw_images"]:
        img_data = base64.b64decode(b64)
        img = Image.open(io.BytesIO(img_data)).convert('RGB')
        arr = np.array(img)
        # CLAHE to fix lighting in the dark cup
        lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        final = cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB)
        _, buff = cv2.imencode('.jpg', cv2.cvtColor(final, cv2.COLOR_RGB2BGR))
        processed.append(base64.b64encode(buff).decode('utf-8'))
    return {"clean_images": processed}

def generation_node(state: CoffeeReaderState):
    api_key = os.environ.get("GROQ_API_KEY")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash", # High speed, great vision
        google_api_key=os.environ.get("GEMINI_API_KEY"),
        temperature=0.7
    )
    # Gemini uses a list of parts for multimodal content
    message_content = [
        {"type": "text", "text": "Interpret these coffee cup views. Focus on Arabic symbols (Snakes, Eyes) and social life."}
    ]

    # Add the images
    for b64_img in state["clean_images"]:
        message_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}
        })

    message = HumanMessage(content=message_content)
    res = llm.invoke([message])

    return {"reading": res.content, "iteration_count": state.get("iteration_count", 0) + 1}

def critique_node(state: CoffeeReaderState):
    # Only critique once to save tokens/time
    if state["iteration_count"] >= 2: return {"critique_notes": None}
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.environ.get("GEMINI_API_KEY"),
        temperature=0.1 # Lower temperature for "strict" critiquing
    )
    # ... (Add logic to check if 'reading' matches the 'clean_images')
    return {"critique_notes": None} # For now, we assume valid

# 3. Graph Assembly
workflow = StateGraph(CoffeeReaderState)
workflow.add_node("cleaner", cleaner_node)
workflow.add_node("generator", generation_node)
workflow.add_node("critique", critique_node)

workflow.add_edge(START, "cleaner")
workflow.add_edge("cleaner", "generator")
workflow.add_edge("generator", "critique")
workflow.add_conditional_edges("critique", lambda x: "generator" if x.get("critique_notes") else END)
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
st.set_page_config(page_title="Arabic Coffee Reader", page_icon="☕")
st.title("🔮 The Digital Tasseographer")

col1, col2, col3 = st.columns(3)
with col1: img1 = st.file_uploader("Right Side", type=['jpg', 'png'])
with col2: img2 = st.file_uploader("Left Side", type=['jpg', 'png'])
with col3: img3 = st.file_uploader("Bottom", type=['jpg', 'png'])

if st.button("Reveal My Future") and img1 and img2 and img3:
    raw_imgs = [base64.b64encode(i.read()).decode('utf-8') for i in [img1, img2, img3]]
    with st.spinner("Analyzing the grounds..."):
        config = {"configurable": {"thread_id": "user_1"}}
        result = app.invoke({"raw_images": raw_imgs, "iteration_count": 0}, config=config)
        st.markdown(result["reading"])

Overwriting app.py


In [14]:
import os
import subprocess
from google.colab import userdata
from pyngrok import ngrok

# 1. Map Gemini and Ngrok secrets
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    NGROK_TOKEN = userdata.get('NGROK_AUTH')
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ Gemini & Ngrok Ready")
except Exception as e:
    print(f"❌ Error: {e}")

# 2. Reset and Run
os.system("pkill streamlit")
ngrok.kill()
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

public_url = ngrok.connect(8501).public_url
print(f"🔮 Gemini Coffee Reader Live: {public_url}")

✅ Gemini & Ngrok Ready
🔮 Gemini Coffee Reader Live: https://cocciferous-dioicous-gino.ngrok-free.dev
